# Train Transformer Prompt Injection Detector trên Google Colab

Notebook này fine-tune Transformer cho bài toán phát hiện Prompt Injection bằng dataset `jayavibhav/prompt-injection`.

Mục tiêu chính:
- Mount Google Drive để lưu dataset split, checkpoint, model output và reports.
- Ưu tiên full fine-tune `xlm-roberta-base` trước, không freeze encoder.
- Có thể train tiếp `roberta-base` và `distilbert-base-uncased` nếu bật trong cấu hình.
- Evaluation đầy đủ: Accuracy, Precision, Recall, F1, F2, ROC-AUC, PR-AUC/AP, TP/FP/TN/FN.
- Calibrate threshold bằng cách quét 0.01 đến 0.99 và chọn theo F2.

Trước khi chạy: vào `Runtime > Change runtime type > GPU`. Với XLM-RoBERTa nên ưu tiên T4/A100/L4 nếu có.

In [ ]:
# Cell 1 - Mount Google Drive và cấu hình đường dẫn
from pathlib import Path
import os
import platform

IS_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IS_COLAB = True
except Exception:
    drive = None

if IS_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/prompt_injection_detector_colab')
else:
    DRIVE_ROOT = Path.cwd() / 'colab_outputs'

DATASET_DIR = DRIVE_ROOT / 'datasets' / 'processed'
MODEL_DIR = DRIVE_ROOT / 'models'
REPORT_DIR = DRIVE_ROOT / 'reports'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
BACKUP_DIR = DRIVE_ROOT / 'backups'

for directory in [DATASET_DIR, MODEL_DIR, REPORT_DIR, CHECKPOINT_DIR, BACKUP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('IS_COLAB:', IS_COLAB)
print('Python:', platform.python_version())
print('DRIVE_ROOT:', DRIVE_ROOT)
print('MODEL_DIR:', MODEL_DIR)
print('REPORT_DIR:', REPORT_DIR)

In [ ]:
# Cell 2 - Cài dependency cần thiết trên Colab
import sys
import subprocess

if IS_COLAB:
    packages = [
        'datasets',
        'transformers==4.46.3',
        'accelerate==0.34.2',
        'evaluate',
        'scikit-learn',
        'pandas',
        'numpy',
        'joblib',
        'sentencepiece'
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
print('Dependencies are ready.')

In [ ]:
# Cell 3 - Import thư viện và kiểm tra GPU
from datetime import datetime
import json
import math
import shutil
import time
from typing import Any

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# Cell 4 - Cấu hình train
DATASET_NAME = 'jayavibhav/prompt-injection'
RANDOM_STATE = 42

# Full mode dùng toàn bộ dataset. Để smoke test nhanh, đặt ví dụ MAX_ROWS = 5000.
MAX_ROWS: int | None = None

# Ưu tiên train XLM-RoBERTa trước. Bật thêm model nếu Colab còn đủ thời gian/GPU.
RUN_MODELS = {
    'xlm_roberta_v4_colab': True,
    'roberta_v4_colab': False,
    'distilbert_v4_colab': False,
}

MODEL_CONFIGS: dict[str, dict[str, Any]] = {
    'xlm_roberta_v4_colab': {
        'base_model': 'xlm-roberta-base',
        'epochs': 4,
        'learning_rate': 1e-5,
        'batch_size': 4,
        'gradient_accumulation_steps': 8,
        'max_length': 128,
        'weight_decay': 0.01,
        'warmup_ratio': 0.10,
        'gradient_checkpointing': True,
        'metric_for_best_model': 'f2',
    },
    'roberta_v4_colab': {
        'base_model': 'roberta-base',
        'epochs': 3,
        'learning_rate': 2e-5,
        'batch_size': 8,
        'gradient_accumulation_steps': 4,
        'max_length': 128,
        'weight_decay': 0.01,
        'warmup_ratio': 0.10,
        'gradient_checkpointing': True,
        'metric_for_best_model': 'f2',
    },
    'distilbert_v4_colab': {
        'base_model': 'distilbert-base-uncased',
        'epochs': 3,
        'learning_rate': 3e-5,
        'batch_size': 16,
        'gradient_accumulation_steps': 2,
        'max_length': 128,
        'weight_decay': 0.01,
        'warmup_ratio': 0.10,
        'gradient_checkpointing': False,
        'metric_for_best_model': 'f2',
    },
}

RESUME_CHECKPOINT = True
SAVE_TOTAL_LIMIT = 2
EARLY_STOPPING_PATIENCE = 2
FP16 = torch.cuda.is_available()

print('Models selected:', [name for name, enabled in RUN_MODELS.items() if enabled])
print('FP16:', FP16)

In [ ]:
# Cell 5 - Load dataset, validate label mapping, deduplicate và split stratified
def normalize_text(value: Any) -> str:
    return str(value).strip()

def load_and_prepare_dataset() -> pd.DataFrame:
    ds = load_dataset(DATASET_NAME)
    if 'train' in ds:
        df = ds['train'].to_pandas()
    else:
        frames = [split.to_pandas() for split in ds.values()]
        df = pd.concat(frames, ignore_index=True)

    required = {'text', 'label'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {missing}. Columns found: {list(df.columns)}')

    df = df[['text', 'label']].copy()
    df['text'] = df['text'].map(normalize_text)
    df = df[df['text'].str.len() > 0].copy()
    df['label'] = pd.to_numeric(df['label'], errors='raise').astype(int)
    labels = sorted(df['label'].unique().tolist())
    if labels != [0, 1]:
        raise ValueError(f'Label mapping invalid. Expected [0, 1], found {labels}')

    before = len(df)
    df['dedup_key'] = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
    df = df.drop_duplicates(subset=['dedup_key'], keep='first').drop(columns=['dedup_key']).reset_index(drop=True)
    df = df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
    if MAX_ROWS is not None:
        df = df.groupby('label', group_keys=False).apply(
            lambda part: part.sample(
                n=min(len(part), max(1, int(MAX_ROWS * len(part) / len(df)))),
                random_state=RANDOM_STATE,
            )
        ).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

    print('Rows before dedup:', before)
    print('Rows after dedup:', len(df))
    print('Label distribution:', df['label'].value_counts().sort_index().to_dict())
    return df

df = load_and_prepare_dataset()
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df['label'],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df['label'],
)

splits = {'train': train_df, 'validation': val_df, 'test': test_df}
for split_name, split_df in splits.items():
    split_df = split_df.reset_index(drop=True).copy()
    split_df.insert(0, 'id', [f'{split_name}_{i:06d}' for i in range(len(split_df))])
    split_df['source'] = DATASET_NAME
    split_df['split'] = split_name
    output_path = DATASET_DIR / f'hf_prompt_injection_{split_name}.csv'
    split_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    splits[split_name] = split_df
    print(split_name, split_df.shape, split_df['label'].value_counts().sort_index().to_dict(), output_path)

combined_df = pd.concat(splits.values(), ignore_index=True)
combined_path = DATASET_DIR / 'hf_prompt_injection_transformer_ready.csv'
combined_df.to_csv(combined_path, index=False, encoding='utf-8-sig')

dataset_summary = {
    'dataset_name': DATASET_NAME,
    'created_at': datetime.utcnow().isoformat() + 'Z',
    'total_rows_after_dedup': int(len(df)),
    'label_mapping_verified': {'0': 'SAFE/BENIGN', '1': 'PROMPT_INJECTION'},
    'split_rows': {name: int(len(part)) for name, part in splits.items()},
    'split_label_distribution': {name: part['label'].value_counts().sort_index().astype(int).to_dict() for name, part in splits.items()},
    'paths': {name: str(DATASET_DIR / f'hf_prompt_injection_{name}.csv') for name in splits},
    'combined_path': str(combined_path),
}
(DATASET_DIR / 'hf_prompt_injection_summary.json').write_text(json.dumps(dataset_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Combined dataset:', combined_path)

In [ ]:
# Cell 6 - Dataset class, metrics và threshold calibration
class PromptDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer: Any, max_length: int):
        self.texts = frame['text'].astype(str).tolist()
        self.labels = frame['label'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, Any]:
        item = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length,
        )
        item['labels'] = self.labels[index]
        return item

def softmax_positive_scores(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    return probabilities[:, 1]

def compute_trainer_metrics(eval_pred: Any) -> dict[str, float]:
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
        'f2': fbeta_score(labels, preds, beta=2, zero_division=0),
    }

def metrics_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    return {
        'threshold': round(float(threshold), 2),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f2': f2,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def calibrate_thresholds(y_true: np.ndarray, scores: np.ndarray) -> dict[str, Any]:
    candidates = [metrics_at_threshold(y_true, scores, t / 100) for t in range(1, 100)]
    best = max(candidates, key=lambda row: (row['f2'], row['recall'], row['precision'], -row['fp']))
    evaluation_threshold = float(best['threshold'])
    warn_threshold = max(0.30, evaluation_threshold)
    precision_candidates = [row for row in candidates if row['precision'] >= 0.95 and row['threshold'] > warn_threshold]
    if precision_candidates:
        block_threshold = float(min(precision_candidates, key=lambda row: row['threshold'])['threshold'])
    else:
        block_threshold = min(0.95, warn_threshold + 0.20)
    if block_threshold <= warn_threshold:
        block_threshold = min(0.95, warn_threshold + 0.15)
    return {
        'evaluation_threshold': round(evaluation_threshold, 2),
        'runtime_warn_threshold': round(warn_threshold, 2),
        'runtime_block_threshold': round(block_threshold, 2),
        'best_metric': 'f2',
        'selected_metrics': best,
        'candidate_metrics': candidates,
    }

def evaluate_scores(y_true: np.ndarray, scores: np.ndarray, thresholds: dict[str, Any]) -> dict[str, Any]:
    selected = metrics_at_threshold(y_true, scores, thresholds['evaluation_threshold'])
    try:
        roc_auc = roc_auc_score(y_true, scores)
    except Exception:
        roc_auc = None
    try:
        average_precision = average_precision_score(y_true, scores)
    except Exception:
        average_precision = None
    return {
        **selected,
        'roc_auc': roc_auc,
        'average_precision': average_precision,
    }

def runtime_action(score: float, thresholds: dict[str, Any]) -> str:
    if score >= thresholds['runtime_block_threshold']:
        return 'block'
    if score >= thresholds['runtime_warn_threshold']:
        return 'warn'
    return 'allow'

In [ ]:
# Cell 7 - Helper train/evaluate một model
def backup_existing_model(output_dir: Path, model_name: str) -> None:
    if not output_dir.exists():
        return
    timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    backup_path = BACKUP_DIR / f'{model_name}_{timestamp}'
    shutil.copytree(output_dir, backup_path)
    print(f'Backed up existing model: {output_dir} -> {backup_path}')

def make_training_args(output_dir: Path, config: dict[str, Any]) -> TrainingArguments:
    kwargs = dict(
        output_dir=str(output_dir),
        save_strategy='epoch',
        learning_rate=config['learning_rate'],
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=config['batch_size'],
        gradient_accumulation_steps=config['gradient_accumulation_steps'],
        num_train_epochs=config['epochs'],
        weight_decay=config['weight_decay'],
        warmup_ratio=config['warmup_ratio'],
        fp16=FP16,
        logging_steps=100,
        save_total_limit=SAVE_TOTAL_LIMIT,
        load_best_model_at_end=True,
        metric_for_best_model=config['metric_for_best_model'],
        greater_is_better=True,
        report_to='none',
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
    )
    try:
        return TrainingArguments(eval_strategy='epoch', **kwargs)
    except TypeError:
        return TrainingArguments(evaluation_strategy='epoch', **kwargs)

def train_one_model(model_name: str, config: dict[str, Any]) -> dict[str, Any]:
    base_model = config['base_model']
    output_dir = MODEL_DIR / model_name
    checkpoint_dir = CHECKPOINT_DIR / model_name
    backup_existing_model(output_dir, model_name)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=2,
        id2label={0: 'SAFE', 1: 'INJECTION'},
        label2id={'SAFE': 0, 'INJECTION': 1},
    )
    if config.get('gradient_checkpointing'):
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    train_dataset = PromptDataset(splits['train'], tokenizer, config['max_length'])
    val_dataset = PromptDataset(splits['validation'], tokenizer, config['max_length'])
    test_dataset = PromptDataset(splits['test'], tokenizer, config['max_length'])
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = make_training_args(checkpoint_dir, config)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_trainer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    resume_path = get_last_checkpoint(str(checkpoint_dir)) if RESUME_CHECKPOINT and checkpoint_dir.exists() else None
    print('Training', model_name, 'from', base_model, 'resume:', resume_path)
    started = time.perf_counter()
    trainer.train(resume_from_checkpoint=resume_path)
    training_time = time.perf_counter() - started

    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    val_predictions = trainer.predict(val_dataset)
    val_scores = softmax_positive_scores(val_predictions.predictions)
    y_val = splits['validation']['label'].astype(int).to_numpy()
    thresholds = calibrate_thresholds(y_val, val_scores)

    test_predictions = trainer.predict(test_dataset)
    test_scores = softmax_positive_scores(test_predictions.predictions)
    y_test = splits['test']['label'].astype(int).to_numpy()
    test_metrics = evaluate_scores(y_test, test_scores, thresholds)

    result = {
        'model': model_name,
        'base_model': base_model,
        'model_path': str(output_dir),
        'checkpoint_path': str(checkpoint_dir),
        'training_time_seconds': training_time,
        'config': config,
        'thresholds': {
            'evaluation_threshold': thresholds['evaluation_threshold'],
            'runtime_warn_threshold': thresholds['runtime_warn_threshold'],
            'runtime_block_threshold': thresholds['runtime_block_threshold'],
            'best_metric': thresholds['best_metric'],
        },
        'validation_selected_metrics': thresholds['selected_metrics'],
        'test_metrics': test_metrics,
    }

    (output_dir / 'training_metadata.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return {
        'result': result,
        'test_scores': test_scores,
        'test_labels': y_test,
        'test_frame': splits['test'].copy(),
    }

In [ ]:
# Cell 8 - Train các model đã bật
run_outputs: dict[str, Any] = {}
for model_name, enabled in RUN_MODELS.items():
    if not enabled:
        print('Skip', model_name)
        continue
    try:
        run_outputs[model_name] = train_one_model(model_name, MODEL_CONFIGS[model_name])
    except RuntimeError as exc:
        if 'out of memory' in str(exc).lower():
            print('CUDA OOM while training', model_name)
            print('Gợi ý: giảm batch_size xuống 2 hoặc 4, tăng gradient_accumulation_steps, giữ max_length=128, bật gradient_checkpointing.')
            torch.cuda.empty_cache()
        raise

print('Finished models:', list(run_outputs))

In [ ]:
# Cell 9 - Xuất evaluation, error cases, score distribution và thresholds.json
evaluation_rows = []
error_rows = []
score_distribution_rows = []
threshold_payload = {'models': {}}

for model_name, payload in run_outputs.items():
    result = payload['result']
    scores = payload['test_scores']
    labels = payload['test_labels']
    frame = payload['test_frame'].reset_index(drop=True)
    thresholds = result['thresholds']
    metrics = result['test_metrics']
    predictions = (scores >= thresholds['evaluation_threshold']).astype(int)

    evaluation_rows.append({
        'dataset': DATASET_NAME,
        'split': 'test',
        'model': model_name,
        'base_model': result['base_model'],
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1': metrics['f1'],
        'f2': metrics['f2'],
        'roc_auc': metrics['roc_auc'],
        'average_precision': metrics['average_precision'],
        'tn': metrics['tn'],
        'fp': metrics['fp'],
        'fn': metrics['fn'],
        'tp': metrics['tp'],
        'evaluation_threshold': thresholds['evaluation_threshold'],
        'warn_threshold': thresholds['runtime_warn_threshold'],
        'block_threshold': thresholds['runtime_block_threshold'],
        'model_path': result['model_path'],
        'training_time_seconds': result['training_time_seconds'],
    })

    threshold_payload['models'][model_name] = {
        **thresholds,
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1': metrics['f1'],
        'f2': metrics['f2'],
        'tn': int(metrics['tn']),
        'fp': int(metrics['fp']),
        'fn': int(metrics['fn']),
        'tp': int(metrics['tp']),
        'model_path': result['model_path'],
        'calibrated_on': 'validation',
        'evaluated_on': 'test',
    }

    for idx, (truth, pred, score) in enumerate(zip(labels, predictions, scores)):
        if int(truth) == int(pred):
            continue
        error_rows.append({
            'model': model_name,
            'error_type': 'FP' if int(truth) == 0 and int(pred) == 1 else 'FN',
            'id': frame.loc[idx, 'id'],
            'ground_truth_label': int(truth),
            'predicted_label': int(pred),
            'risk_score': float(score),
            'action': runtime_action(float(score), thresholds),
            'text': frame.loc[idx, 'text'],
        })

    for label in [0, 1]:
        label_scores = scores[labels == label]
        buckets = np.minimum((label_scores * 10).astype(int), 9)
        for bucket in range(10):
            score_distribution_rows.append({
                'model': model_name,
                'label': label,
                'score_bin': f'{bucket / 10:.1f}-{(bucket + 1) / 10:.1f}',
                'count': int((buckets == bucket).sum()),
            })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_path = REPORT_DIR / 'transformer_colab_full_evaluation.csv'
evaluation_df.to_csv(evaluation_path, index=False, encoding='utf-8-sig')

error_path = REPORT_DIR / 'error_cases.csv'
pd.DataFrame(error_rows).to_csv(error_path, index=False, encoding='utf-8-sig')

score_distribution_path = REPORT_DIR / 'score_distribution.csv'
pd.DataFrame(score_distribution_rows).to_csv(score_distribution_path, index=False, encoding='utf-8-sig')

threshold_path = MODEL_DIR / 'thresholds.json'
threshold_path.write_text(json.dumps(threshold_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('Evaluation:', evaluation_path)
print('Errors:', error_path)
print('Score distribution:', score_distribution_path)
print('Thresholds:', threshold_path)
display(evaluation_df)

In [ ]:
# Cell 10 - Ghi Markdown reports
def fmt(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    if isinstance(value, float):
        return f'{value:.4f}'
    return str(value)

summary_lines = [
    '# Transformer Colab Training Summary',
    '',
    f'Generated at: `{datetime.utcnow().isoformat()}Z`',
    '',
    '## Dataset',
    '',
    f'- Dataset: `{DATASET_NAME}`',
    f'- Drive root: `{DRIVE_ROOT}`',
    f'- Train/validation/test split files: `{DATASET_DIR}`',
    f'- Label mapping: `0 = SAFE/BENIGN`, `1 = PROMPT INJECTION`',
    '',
    '## Model Evaluation',
    '',
    '| Model | Accuracy | Precision | Recall | F1 | F2 | ROC-AUC | AP/PR-AUC | TP | FP | TN | FN | Eval Th | Warn | Block |',
    '| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for _, row in evaluation_df.iterrows():
    summary_lines.append('| ' + ' | '.join([
        row['model'], fmt(row['accuracy']), fmt(row['precision']), fmt(row['recall']), fmt(row['f1']), fmt(row['f2']),
        fmt(row['roc_auc']), fmt(row['average_precision']), str(row['tp']), str(row['fp']), str(row['tn']), str(row['fn']),
        fmt(row['evaluation_threshold']), fmt(row['warn_threshold']), fmt(row['block_threshold']),
    ]) + ' |')
summary_lines.extend([
    '',
    '## Output files',
    '',
    f'- `transformer_colab_full_evaluation.csv`: `{evaluation_path}`',
    f'- `error_cases.csv`: `{error_path}`',
    f'- `score_distribution.csv`: `{score_distribution_path}`',
    f'- `models/thresholds.json`: `{threshold_path}`',
    '',
    '## Notes',
    '',
    '- Threshold được calibrate trên validation split và evaluate trên test split.',
    '- Evaluation threshold dùng cho report; warn/block threshold dùng cho runtime.',
    '- Nếu Colab bị OOM, giảm batch_size, giữ max_length=128, tăng gradient_accumulation_steps và bật gradient_checkpointing.',
])

summary_path = REPORT_DIR / 'transformer_colab_training_summary.md'
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')

threshold_lines = [
    '# Threshold Summary',
    '',
    'Threshold được chọn bằng cách quét từ 0.01 đến 0.99 và tối ưu F2 trên validation split.',
    '',
    '| Model | Eval Threshold | Warn Threshold | Block Threshold | Precision | Recall | F1 | F2 | TN | FP | FN | TP |',
    '| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for _, row in evaluation_df.iterrows():
    threshold_lines.append('| ' + ' | '.join([
        row['model'], fmt(row['evaluation_threshold']), fmt(row['warn_threshold']), fmt(row['block_threshold']),
        fmt(row['precision']), fmt(row['recall']), fmt(row['f1']), fmt(row['f2']),
        str(row['tn']), str(row['fp']), str(row['fn']), str(row['tp']),
    ]) + ' |')
threshold_summary_path = REPORT_DIR / 'threshold_summary.md'
threshold_summary_path.write_text('\n'.join(threshold_lines), encoding='utf-8')

print('Summary:', summary_path)
print('Threshold summary:', threshold_summary_path)

## Cách tải model về local project

Sau khi train xong, trong Google Drive sẽ có thư mục:

```text
MyDrive/prompt_injection_detector_colab/models/<model_name>/
MyDrive/prompt_injection_detector_colab/models/thresholds.json
MyDrive/prompt_injection_detector_colab/reports/
```

Tải thư mục model về local project và đặt vào:

```text
models/transformers/<model_name>/
```

Sau đó copy `models/thresholds.json` hoặc merge phần model mới vào `models/transformer_thresholds.json` tùy pipeline runtime local đang dùng.